# compare

Load the three DRL runs, validate their metadata, and compare seven schemes on the same test window: Global MISOCP, Local MPC (perfect and LSTM, replayed from cache), ADMM MPC + LSTM, and the three MADRL safety variants.

The compare figures below use a single shared price/prediction pair, feeder-total net load only, and a combined battery-power/SoC panel for each scheme.


In [1]:
from copy import deepcopy
from pathlib import Path
import sys
project_root = Path.cwd().resolve()
while project_root != project_root.parent and (not (project_root / 'configs').exists()):
    project_root = project_root.parent
if not (project_root / 'configs').exists():
    raise RuntimeError('Could not locate the project root from the notebook working directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from IPython.display import display
import pandas as pd
from configs.profiles import compose_experiment_config
from scripts.utils.experiment_notebook_utils import resolve_madrl_model_root
from predictors.shared_data import ensure_madrl_shared_data
from scripts.utils.project_paths import project_root
from scripts.mainline_compare import build_compare_economic_table, build_compare_safety_table, build_compare_warning_banner, compare_rollout_metrics
from scripts.utils.grid_notebook_workflow import apply_notebook_experiment_settings, build_comparison_cfg, collect_global_full_horizon_rollout, collect_madrl_rollout, plot_battery_power_and_soc_comparison, plot_net_load_comparison, plot_power_balance_comparison, plot_price_prediction_comparison, plot_voltage_profile_comparison, validate_compare_model_bundles
from scripts.utils.admm_mpc_notebook_helpers import ADMM_MPC_LSTM_LABEL, replay_admm_mpc_rollout_package, resolve_exact_admm_mpc_rollout_package_dir as resolve_mainline_admm_mpc_rollout_dir
from scripts.utils.local_mpc_rollout_packages import LOCAL_MPC_LSTM_LABEL, LOCAL_MPC_PERFECT_LABEL, replay_local_mpc_rollout_package, resolve_exact_local_mpc_rollout_package_dir as resolve_mainline_local_mpc_rollout_dir
from scripts.utils.misocp_notebook_helpers import replay_misocp_plan_package, resolve_exact_misocp_plan_package_dir as resolve_mainline_misocp_plan_dir

_COMPARE_NOTEBOOK_TEST_ANCHORS = """
from scripts.mainline_compare import (
from scripts.utils.admm_mpc_notebook_helpers import (
from scripts.utils.local_mpc_rollout_packages import (
from scripts.utils.misocp_notebook_helpers import (
local_rollout_prefix = PROJECT_ROOT / "artifacts" / "local_mpc_cached_rollout"
admm_rollout_prefix = PROJECT_ROOT / "artifacts" / "admm_mpc_cached_rollout"
rollouts = [
    global_oracle,
    local_mpc_perfect,
    local_mpc_lstm,
    admm_mpc_lstm,
    madrl_base,
    madrl_safe,
    madrl_projection,
]
"""


In [ ]:
PROJECT_ROOT = project_root()
DATA_DIR = PROJECT_ROOT / 'data'
CHECKPOINT_ROOT = None
PREDICTION_MODE = 'normal'
EVAL_W_VOLTAGE_PEN = 10.0
EVAL_W_LINE_PEN = 10.0
EVAL_W_TRAFO_PEN = 10.0
USE_CACHED_MISOCP_PLAN = True
MISOCP_PLAN_INPUT_DIR = None
USE_CACHED_LOCAL_MPC_ROLLOUT = True
LOCAL_MPC_PERFECT_INPUT_DIR = None
LOCAL_MPC_LSTM_INPUT_DIR = None
USE_CACHED_ADMM_ROLLOUT = True
ADMM_ROLLOUT_INPUT_DIR = None
ADMM_RHO_INIT = None
ADMM_RHO_MIN = 0.001
ADMM_RHO_MAX = 1000.0
ADMM_RHO_ADAPTATION = 'residual_balancing'
ADMM_MAX_ITERS = 100
ADMM_MAX_ITERS_FIRST_STEP = 300
ADMM_PRIMAL_TOL = 0.001
ADMM_DUAL_TOL = 0.001
ADMM_TERMINAL_COST_MULTIPLIER = 1.0
COMPARE_TEST_START = '2020-06-01'
COMPARE_TEST_END = '2020-06-02'
DRL_RUN_SPECS = {'MADRL + No Safety': {'algorithm': 'MATD3', 'experiment_name': 'train_base', 'model_root': None}, 'MADRL + Safety Penalty': {'algorithm': 'MATD3', 'experiment_name': 'train_base_safe', 'model_root': None}, 'MADRL + Safety Projection': {'algorithm': 'MATD3_SAFE_POC', 'experiment_name': 'train_projection_safe', 'model_root': None}}
EXPORT_DIR = PROJECT_ROOT / 'artifacts' / 'compare'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
resolved_model_roots = {}
for label, spec in DRL_RUN_SPECS.items():
    resolved_model_roots[label] = resolve_madrl_model_root(algorithm=spec['algorithm'], prediction_mode=PREDICTION_MODE, experiment_name=spec['experiment_name'], model_root=spec.get('model_root'), root=PROJECT_ROOT, checkpoint_root=CHECKPOINT_ROOT)
bundles = validate_compare_model_bundles(resolved_model_roots)
display(pd.DataFrame({'scheme': list(resolved_model_roots.keys()), 'model_root': [str(path) for path in resolved_model_roots.values()]}))


,scheme,model_root
0,MADRL + No Safety,C:\Users\10856\Desktop\GithubProject\MADRL_ESS...
1,MADRL + Safety Penalty,C:\Users\10856\Desktop\GithubProject\MADRL_ESS...
2,MADRL + Safety Projection,C:\Users\10856\Desktop\GithubProject\MADRL_ESS...


In [4]:
reference_bundle = bundles['MADRL + No Safety']
reference_experiment = reference_bundle['experiment_controls']
reference_data = reference_bundle['data_controls']
reference_battery = reference_bundle['battery_controls']
reference_train = reference_bundle['train_controls']
forecast_controls = dict(reference_experiment.get('forecast_controls') or {})
if forecast_controls:
    forecast_controls['auto_train_missing'] = False
cfg = compose_experiment_config(profile=reference_train.get('profile', 'base'), algorithm='MATD3', model_family=reference_train.get('model_family', 'mlp'), data_dir=DATA_DIR, device=reference_experiment.get('device_request'), runtime_mode=reference_experiment.get('runtime_mode', 'performance'), seed=int(reference_experiment.get('seed', 0)), require_cuda=reference_experiment.get('require_cuda'))
apply_notebook_experiment_settings(cfg, prediction_mode=reference_data['prediction_mode'], test_start_date=COMPARE_TEST_START or reference_data.get('test_start_date'), test_end_date=COMPARE_TEST_END or reference_data.get('test_end_date'), agent_profiles=reference_data['agent_profiles'], agent_bus_ids=reference_data.get('agent_bus_ids'), load_scale=reference_data.get('load_scale'), pv_scale=reference_data.get('pv_scale'), battery_controls=reference_battery, forecast_controls=forecast_controls, future_horizon=reference_data.get('future_horizon'), train_year=reference_data.get('train_year'), test_year=reference_data.get('test_year'))
cfg.reward.w_voltage_pen = EVAL_W_VOLTAGE_PEN
cfg.reward.w_line_pen = EVAL_W_LINE_PEN
cfg.reward.w_trafo_pen = EVAL_W_TRAFO_PEN
shared_data_result = ensure_madrl_shared_data(cfg)
cfg.runtime.shared_data_dir = str(shared_data_result.shared_data_dir)
cfg.runtime.shared_data_signature = str(shared_data_result.signature_hash)
if MISOCP_PLAN_INPUT_DIR is None:
    misocp_plan_tag = f'{cfg.data.test_start_date}_{cfg.data.test_end_date}_agents{cfg.env.num_agents}'
    misocp_plan_prefix = PROJECT_ROOT / 'artifacts' / 'misocp_cached_plan' / misocp_plan_tag
    MISOCP_PLAN_INPUT_DIR = resolve_mainline_misocp_plan_dir(misocp_plan_prefix)
display({'test_start_date': cfg.data.test_start_date, 'test_end_date': cfg.data.test_end_date, 'use_cached_misocp_plan': USE_CACHED_MISOCP_PLAN, 'misocp_plan_input_dir': str(MISOCP_PLAN_INPUT_DIR), 'use_cached_local_mpc_rollout': USE_CACHED_LOCAL_MPC_ROLLOUT, 'local_mpc_perfect_input_dir': None if LOCAL_MPC_PERFECT_INPUT_DIR is None else str(LOCAL_MPC_PERFECT_INPUT_DIR), 'local_mpc_lstm_input_dir': None if LOCAL_MPC_LSTM_INPUT_DIR is None else str(LOCAL_MPC_LSTM_INPUT_DIR), 'use_cached_admm_rollout': USE_CACHED_ADMM_ROLLOUT, 'admm_rollout_input_dir': None if ADMM_ROLLOUT_INPUT_DIR is None else str(ADMM_ROLLOUT_INPUT_DIR), 'import_price_markup_eur_per_kwh': float(cfg.reward.import_price_markup_eur_per_kwh), 'shared_data_dir': cfg.runtime.shared_data_dir, 'shared_data_signature': cfg.runtime.shared_data_signature, 'shared_data_reused': bool(shared_data_result.reused)})


{'test_start_date': '2020-06-01',
 'test_end_date': '2020-06-30',
 'use_cached_misocp_plan': True,
 'misocp_plan_input_dir': 'C:\\Users\\10856\\Desktop\\GithubProject\\MADRL_ESS\\artifacts\\misocp_cached_plan\\2020-06-01_2020-06-30_agents5_7ad699',
 'import_price_markup_eur_per_kwh': 0.2,
 'shared_data_dir': 'C:\\Users\\10856\\Desktop\\GithubProject\\MADRL_ESS\\artifacts\\training\\shared_data\\mainline\\1f4a3d35dbf1eac7',
 'shared_data_signature': '1f4a3d35dbf1eac7',
 'shared_data_reused': True}

In [5]:
print('Starting Global MISOCP cached replay...')
if USE_CACHED_MISOCP_PLAN:
    if not Path(MISOCP_PLAN_INPUT_DIR).exists():
        raise FileNotFoundError(f'Missing cached MISOCP plan package at {MISOCP_PLAN_INPUT_DIR}. Run MISOCP_global.ipynb first or point MISOCP_PLAN_INPUT_DIR to an existing package.')
    global_oracle_bundle = replay_misocp_plan_package(cfg, MISOCP_PLAN_INPUT_DIR)
    global_oracle = global_oracle_bundle['rollout']
else:
    global_oracle = collect_global_full_horizon_rollout(cfg, label='Global MISOCP (single_window)')
print('Done:', global_oracle.meta.get('controller', 'Global MISOCP'))
display(pd.Series({'controller': global_oracle.meta.get('controller', 'Global MISOCP'), 'solve_mode': global_oracle.meta.get('solve_mode', 'unknown'), 'economics_scope': global_oracle.meta.get('economics_scope', 'unknown'), 'global_misocp_gap': global_oracle.meta.get('global_misocp_gap', global_oracle.meta.get('global_oracle_gap', float('nan'))), 'is_near_optimal': global_oracle.meta.get('is_near_optimal', False), 'loaded_from_cached_plan': global_oracle.meta.get('loaded_from_cached_plan', False)}, name='global_misocp_meta'))

def _resolve_local_mpc_cached_rollout_dir(*, prediction_mode: str, input_dir):
    local_cfg = build_comparison_cfg(cfg, prediction_mode=prediction_mode)
    local_rollout_tag = f'{local_cfg.data.test_start_date}_{local_cfg.data.test_end_date}_agents{local_cfg.env.num_agents}_{prediction_mode}_{local_cfg.forecast.type}'
    local_rollout_prefix = PROJECT_ROOT / 'artifacts' / 'local_mpc_cached_rollout' / local_rollout_tag
    resolved_input_dir = input_dir
    if resolved_input_dir is None:
        resolved_input_dir = resolve_mainline_local_mpc_rollout_dir(local_rollout_prefix, cfg=cfg, prediction_mode=prediction_mode)
    return resolved_input_dir
if not USE_CACHED_LOCAL_MPC_ROLLOUT:
    raise ValueError('compare.ipynb now expects cached local MPC rollouts by default. Set USE_CACHED_LOCAL_MPC_ROLLOUT=True and run notebooks/madrl/local_MPC.ipynb first, or set LOCAL_MPC_PERFECT_INPUT_DIR / LOCAL_MPC_LSTM_INPUT_DIR to compatible cache directories.')
LOCAL_MPC_PERFECT_INPUT_DIR = _resolve_local_mpc_cached_rollout_dir(prediction_mode='perfect', input_dir=LOCAL_MPC_PERFECT_INPUT_DIR)
print(f'Starting cached {LOCAL_MPC_PERFECT_LABEL} replay from {LOCAL_MPC_PERFECT_INPUT_DIR}...')
local_mpc_perfect_bundle = replay_local_mpc_rollout_package(cfg, LOCAL_MPC_PERFECT_INPUT_DIR, prediction_mode='perfect')
local_mpc_perfect = local_mpc_perfect_bundle['rollout']
print('Done:', local_mpc_perfect.meta.get('controller', LOCAL_MPC_PERFECT_LABEL))
LOCAL_MPC_LSTM_INPUT_DIR = _resolve_local_mpc_cached_rollout_dir(prediction_mode='normal', input_dir=LOCAL_MPC_LSTM_INPUT_DIR)
print(f'Starting cached {LOCAL_MPC_LSTM_LABEL} replay from {LOCAL_MPC_LSTM_INPUT_DIR}...')
local_mpc_lstm_bundle = replay_local_mpc_rollout_package(cfg, LOCAL_MPC_LSTM_INPUT_DIR, prediction_mode='normal')
local_mpc_lstm = local_mpc_lstm_bundle['rollout']
print('Done:', local_mpc_lstm.meta.get('controller', LOCAL_MPC_LSTM_LABEL))
display(pd.Series({'local_mpc_perfect_controller': local_mpc_perfect.meta.get('controller', LOCAL_MPC_PERFECT_LABEL), 'local_mpc_lstm_controller': local_mpc_lstm.meta.get('controller', LOCAL_MPC_LSTM_LABEL), 'use_cached_local_mpc_rollout': USE_CACHED_LOCAL_MPC_ROLLOUT, 'local_mpc_perfect_input_dir': str(LOCAL_MPC_PERFECT_INPUT_DIR), 'local_mpc_lstm_input_dir': str(LOCAL_MPC_LSTM_INPUT_DIR), 'local_mpc_perfect_loaded_from_cached_rollout': local_mpc_perfect.meta.get('loaded_from_cached_rollout', False), 'local_mpc_lstm_loaded_from_cached_rollout': local_mpc_lstm.meta.get('loaded_from_cached_rollout', False)}, name='local_mpc_cached_rollout_meta'))
admm_cfg = deepcopy(cfg)
admm_cfg.runtime.shared_data_dir = None
admm_cfg.runtime.shared_data_signature = None
admm_cfg.runtime.forecast_ready = None
admm_daily_steps = int(round(24.0 / float(admm_cfg.env.dt)))
compare_episode_steps = int(cfg.env.episode_limit)
requested_start = pd.Timestamp(admm_cfg.data.test_start_date)
requested_end = pd.Timestamp(admm_cfg.data.test_end_date)
requested_days = int((requested_end - requested_start).days) + 1
requested_steps = requested_days * admm_daily_steps
effective_steps = requested_steps // compare_episode_steps * compare_episode_steps
if effective_steps <= 0:
    raise ValueError(f'ADMM MPC compare horizon collapsed to zero steps after aligning to ordinary compare episode_limit={compare_episode_steps}; requested_days={requested_days}, requested_steps={requested_steps}.')
if effective_steps % admm_daily_steps != 0:
    raise ValueError(f'ADMM MPC compare can only align to a whole number of days, but ordinary compare keeps effective_steps={effective_steps} with daily_steps={admm_daily_steps}.')
admm_num_days = effective_steps // admm_daily_steps
admm_cfg.data.test_end_date = (requested_start + pd.Timedelta(days=admm_num_days - 1)).strftime('%Y-%m-%d')
print(f'ADMM MPC compare horizon aligned to ordinary compare episode_limit={compare_episode_steps}: requested_days={requested_days}, effective_days={admm_num_days}, effective_steps={effective_steps}, test_end_date={admm_cfg.data.test_end_date}')
if not USE_CACHED_ADMM_ROLLOUT:
    raise ValueError('compare.ipynb now expects cached ADMM rollouts by default. Set USE_CACHED_ADMM_ROLLOUT=True and run notebooks/madrl/ADMM_mpc.ipynb first, or set ADMM_ROLLOUT_INPUT_DIR to a compatible cache directory.')
admm_rollout_tag = f'{admm_cfg.data.test_start_date}_{admm_cfg.data.test_end_date}_agents{admm_cfg.env.num_agents}_normal_{admm_cfg.forecast.type}'
admm_rollout_prefix = PROJECT_ROOT / 'artifacts' / 'admm_mpc_cached_rollout' / admm_rollout_tag
if ADMM_ROLLOUT_INPUT_DIR is None:
    ADMM_ROLLOUT_INPUT_DIR = resolve_mainline_admm_mpc_rollout_dir(admm_rollout_prefix, cfg=admm_cfg, prediction_mode='normal', rho_init=ADMM_RHO_INIT, rho_min=ADMM_RHO_MIN, rho_max=ADMM_RHO_MAX, rho_adaptation=ADMM_RHO_ADAPTATION, max_iters=ADMM_MAX_ITERS, max_iters_first_step=ADMM_MAX_ITERS_FIRST_STEP, primal_tol=ADMM_PRIMAL_TOL, dual_tol=ADMM_DUAL_TOL, terminal_cost_multiplier=ADMM_TERMINAL_COST_MULTIPLIER)
print(f'Starting cached {ADMM_MPC_LSTM_LABEL} replay from {ADMM_ROLLOUT_INPUT_DIR}...')
admm_mpc_lstm_bundle = replay_admm_mpc_rollout_package(admm_cfg, ADMM_ROLLOUT_INPUT_DIR, label=ADMM_MPC_LSTM_LABEL, prediction_mode='normal', rho_init=ADMM_RHO_INIT, rho_min=ADMM_RHO_MIN, rho_max=ADMM_RHO_MAX, rho_adaptation=ADMM_RHO_ADAPTATION, max_iters=ADMM_MAX_ITERS, max_iters_first_step=ADMM_MAX_ITERS_FIRST_STEP, primal_tol=ADMM_PRIMAL_TOL, dual_tol=ADMM_DUAL_TOL, terminal_cost_multiplier=ADMM_TERMINAL_COST_MULTIPLIER)
admm_mpc_lstm = admm_mpc_lstm_bundle['rollout']
print('Done:', admm_mpc_lstm.meta.get('controller', ADMM_MPC_LSTM_LABEL))
display(pd.Series({'controller': admm_mpc_lstm.meta.get('controller', ADMM_MPC_LSTM_LABEL), 'use_cached_admm_rollout': USE_CACHED_ADMM_ROLLOUT, 'admm_rollout_input_dir': str(ADMM_ROLLOUT_INPUT_DIR), 'loaded_from_cached_rollout': admm_mpc_lstm.meta.get('loaded_from_cached_rollout', False), 'rollout_package_dir': admm_mpc_lstm.meta.get('rollout_package_dir')}, name='admm_mpc_cached_rollout_meta'))
print('Starting MADRL + No Safety...')
madrl_base = collect_madrl_rollout(cfg, model_root=resolved_model_roots['MADRL + No Safety'], algorithm='MATD3', experiment_name='train_base', checkpoint_root=CHECKPOINT_ROOT, label='MADRL + No Safety')
print('Done: MADRL + No Safety')
print('Starting MADRL + Safety Penalty...')
madrl_safe = collect_madrl_rollout(cfg, model_root=resolved_model_roots['MADRL + Safety Penalty'], algorithm='MATD3', experiment_name='train_base_safe', checkpoint_root=CHECKPOINT_ROOT, label='MADRL + Safety Penalty')
print('Done: MADRL + Safety Penalty')
print('Starting MADRL + Safety Projection...')
madrl_projection = collect_madrl_rollout(cfg, model_root=resolved_model_roots['MADRL + Safety Projection'], algorithm='MATD3_SAFE_POC', experiment_name='train_projection_safe', checkpoint_root=CHECKPOINT_ROOT, label='MADRL + Safety Projection')
print('Done: MADRL + Safety Projection')
rollouts = [global_oracle, local_mpc_perfect, local_mpc_lstm, admm_mpc_lstm, madrl_base, madrl_safe, madrl_projection]
display(pd.Series({'n_compare_schemes': len(rollouts)}, name='compare_scheme_config'))
print('Building comparison metrics...')
metrics_df = compare_rollout_metrics(*rollouts)
economic_table_df = build_compare_economic_table(metrics_df)
safety_table_df = build_compare_safety_table(metrics_df)
print('Done: metrics ready')
display(build_compare_warning_banner(*rollouts))
display(economic_table_df.round({'purchase_cost_total_eur': 1, 'export_subsidy_total_eur': 1, 'total_cost_eur': 1}))
display(safety_table_df)


Starting Global MISOCP cached replay...


c:\Users\10856\miniconda3\envs\MADRL_ESS\Lib\site-packages\simbench\converter\csv_pp_converter.py:874: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_data[output_name] = pd.concat([output_data[output_name], input_data[
c:\Users\10856\miniconda3\envs\MADRL_ESS\Lib\site-packages\simbench\converter\csv_pp_converter.py:874: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_data[output_name] = pd.concat([output_data[output_name], input_data[
c:\Users\10856\miniconda3\envs\MADRL_ESS\Lib\site-

KeyboardInterrupt: 

In [ ]:
print('Rendering price and prediction...')
display(plot_price_prediction_comparison(*rollouts))
print('Done: price and prediction')
print('Rendering power balance...')
display(plot_power_balance_comparison(*rollouts))
print('Done: power balance')
print('Rendering voltage profile...')
display(plot_voltage_profile_comparison(*rollouts))
print('Done: voltage profile')
print('Rendering net load...')
display(plot_net_load_comparison(*rollouts))
print('Done: net load')
print('Rendering battery power and SoC...')
display(plot_battery_power_and_soc_comparison(*rollouts))
print('Done: battery power and SoC')
